# Welcome to Week 4 - LangChain and LangGraph

## Lab 1: Abstraction Levels and the building blocks

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The LangChain docs</h2>
            <span style="color:#00bfff;">The documentation lives at <a href="https://docs.langchain.com/oss/python/">https://docs.langchain.com/oss/python/</a>. It is worth a browse. Watch out for older 0.x material that still floats around the web, because the modern API is quite different.
            </span>
        </td>
    </tr>
</table>

## The four levels of abstraction

LangChain and LangGraph form 4 levels of abstraction, with each one built on the ones before. The terminology is confusing because 'LangChain' appears in a few places..

| Layer | Packages | What it gives you | What you control |
|---|---|---|---|
| 1. Building blocks | `langchain-core` + `langchain-openai` | chat models, the `@tool` decorator, messages, structured output | everything, including the tool loop by hand |
| 2. Orchestration | `langgraph` | a graph of steps, with state, memory and checkpointing | the control flow (you design the graph) |
| 3. Agent | `langchain` (`create_agent`) | the standard agent loop, prebuilt | just model, tools and a prompt |
| 4. Harness | `deepagents` (`create_deep_agent`) | an opinionated harness with planning, sub-agents and a filesystem | your intent |


## Order of play for this week:

DAY 1 (today): The building blocks  
DAY 2: LangGraph  
DAY 3: LangChain create_agent  
DAY 4: Deep Agents  
DAY 5: The Sidekick project  

## Today is Layer 1: the building blocks

This is where LangChain began: an abstraction layer, not unlike LiteLLM - but a more heavyweight version.

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True)

True

### Model Calls to OpenAI models and models from other LLM providers

`ChatOpenAI` is the abstraction around OpenAI calls, and there are similar packages for other LLM providers.

We call `invoke` with a prompt; this is a key method in LangChain.

In [3]:
LLM = ChatOpenAI(model="gpt-5.4-mini")
Message = "In one sentence, what is the Swiss Ephemeris"
Response = LLM.invoke(Message)
print (Response.content)

LLM = ChatOpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai",
                api_key=os.getenv("GOOGLE_API_KEY"), model="gemini-3.1-flash-lite")
print (LLM.invoke("What is Default Mode Network in the human brain? Answer in under 100 words").content)

The Swiss Ephemeris is a high-precision astronomical calculation library used to compute planetary and celestial positions for astrology and astronomy applications.
The **Default Mode Network (DMN)** is a collection of interconnected brain regions—primarily the medial prefrontal cortex, posterior cingulate cortex, and angular gyrus—that are most active when the mind is at rest and not focused on the outside world.

It is associated with "internal" mental processes, including daydreaming, reflecting on the past, envisioning the future, thinking about others, and mind-wandering. When you concentrate on an external task, the DMN typically deactivates, allowing task-positive networks to take over. It is essential for self-referential thought and maintaining a stable sense of self.


### Streaming

For a live, token by token feel, swap `invoke` for `stream` and loop over the chunks

In [4]:
for Chunk in LLM.stream("Tell me a haiku about astrology"):
    print (Chunk.content, end="", flush=True)

Stars dance in the dark,
Mapping out our hidden paths,
Written in the sky.

### Messages

LangChain comes with abstractions around SystemMessage, HumanMessage, AIMessage, although you can use the usual list-of-dicts instead.

In [6]:
Messages = [
    SystemMessage("You are an assistant who replies in a very terse manner"),
    HumanMessage("What is the capital of Mauritius")
]
Result = LLM.invoke(Messages)
print ("Type of Result: ", type(Result))
print (Result.content)

## The exact same call can be made using plain dictionaries
MessageDicts = [
    {"role":"system", "content":"You are an assistant who replies in a very terse manner"},
    {"role":"user", "content":"What is the capital of Democatric Republic of Congo"}
]
print (LLM.invoke(MessageDicts).content)

Type of Result:  <class 'langchain_core.messages.ai.AIMessage'>
Port Louis.
Kinshasa.


### Tools with the `@tool` decorator

A tool is a Python function the model is allowed to call. The modern way to make one is the `@tool` decorator. Your docstring becomes the description the model reads, and your type hints become the argument schema, just like `@function_tool` with OpenAI Agents SDK.

This replaces the older `Tool(...)` wrapper from earlier versions of LangChain.

In [12]:
@tool
def GetSharePrice(Symbol: str) -> float:
    """7Return the current share prices for a given ticker symbol"""
    SimulatedPrices = {'AAPL': 241.5, "GOOGL": 168.2, "AMZN": 198.0}
    return SimulatedPrices.get(Symbol.upper(), 0.0)

print ("Name: ", GetSharePrice.name)
print ("Description: ", GetSharePrice.description)
print ("Arguments: ", GetSharePrice.args)
print ("Results of tool called directly: ",GetSharePrice.invoke({"Symbol":"AAPL"}))

Name:  GetSharePrice
Description:  7Return the current share prices for a given ticker symbol
Arguments:  {'Symbol': {'title': 'Symbol', 'type': 'string'}}
Results of tool called directly:  241.5


### Giving tools to the model

This is a bit clunky; redemption will come in Day 3 when we use the create_agent features,

For now, we have to write the loop ourselves, rather like we did in Week 1.

The first step is to bind the tools to the model with `bind_tools`. Now when we invoke, the model may come back not with an answer but with a request to run a tool. That request shows up in `.tool_calls`.

In [16]:
LLM = ChatOpenAI(model="gpt-5.4-nano")
LLMWithTools = LLM.bind_tools([GetSharePrice])
Response = LLMWithTools.invoke("What is the share price of Amazon?")
print ("Type of response: ", type(Response))
print ("Content: ", repr(Response.content))
## The repr funcion, when used for a string, wraps the value in quotes,
## and escapes special characters like \n,\t so they're visible rather than interpreted
print ("Tool calls: ", Response.tool_calls)

Type of response:  <class 'langchain_core.messages.ai.AIMessage'>
Content:  ''
Tool calls:  [{'name': 'GetSharePrice', 'args': {'Symbol': 'AMZN'}, 'id': 'call_yoQNDRXfLmlxtVDQGNgzlOQn', 'type': 'tool_call'}]


### Running the tool loop by hand

So now we need to write a little loop, quite similar to Week 1

In [18]:
# Start the conversation and keep the model's tool request in the history
Conversation = [HumanMessage("What is the share price of Amazon?")]
AIResponse = LLMWithTools.invoke(Conversation)
Conversation.append(AIResponse)

## Run each requested tool and add its result as a ToolMessage
if AIResponse.tool_calls:
    for ToolCall in AIResponse.tool_calls:
        if ToolCall["name"] == "GetSharePrice":
            Result = GetSharePrice.invoke(ToolCall["args"])
            print (Result, "\n")
            Conversation.append(ToolMessage(content=str(Result), tool_call_id = ToolCall["id"]))

## If we invoke the function again now, the model can see the tool has been called
print (LLMWithTools.invoke(Conversation).content)

198.0 

Amazon (AMZN) share price is **$198.00**.


### Structured output

Similar to OpenAI Agents SDK, we can require the model to respond with a Pydantic subclass

In [23]:
class Company(BaseModel):
    name: str = Field("Name of the company")
    ticker: str = Field(description="The stock ticker symbol")
    foundation_year: str = Field(description="The year the company was founded")

StructuredLLM = LLM.with_structured_output(Company)
Result = StructuredLLM.invoke("Tell me about the Indian company Marico")
print (Result,"\n")
print ("Ticker: ", Result.ticker)

name='Marico Limited' ticker='MARICO' foundation_year='1988' 

Ticker:  MARICO


## That is Layer 1

It's like a more rich and more involved version of LiteLLM.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Write a second tool of your own, perhaps one that looks up a fictional weather report for a city. Bind both tools to the model and ask a question that needs both, then run the tool loop by hand until the model gives a final answer. It will feel more gritty than OpenAI Agents SDK or CrewAI.
            We will fix that by Day 3!
            </span>
        </td>
    </tr>
</table>